Salaries Differences Calculates the difference between the highest salaries in the marketing and engineering departments. Output just the absolute difference in salaries. Tables db_employee and db_dept

In [0]:
data_emp = [
    (1, "John", "Doe", 60000, 1),
    (2, "Jane", "Smith", 80000, 2),
    (3, "Sam", "Brown", 75000, 1),
    (4, "Lisa", "Ray", 90000, 2),
    (5, "Tom", "Hanks", 50000, 1)
]

data_dept = [
    (1, "marketing"),
    (2, "engineering")
]

In [0]:
column1 = ['emp_id','name','surname','salary','dept_id']
column2 = ['dept_id','department']

In [0]:
df_emp = spark.createDataFrame(data_emp,column1)
df_emp.show()

+------+----+-------+------+-------+
|emp_id|name|surname|salary|dept_id|
+------+----+-------+------+-------+
|     1|John|    Doe| 60000|      1|
|     2|Jane|  Smith| 80000|      2|
|     3| Sam|  Brown| 75000|      1|
|     4|Lisa|    Ray| 90000|      2|
|     5| Tom|  Hanks| 50000|      1|
+------+----+-------+------+-------+



In [0]:
df_dept = spark.createDataFrame(data_dept,column2)
df_dept.show()

+-------+-----------+
|dept_id| department|
+-------+-----------+
|      1|  marketing|
|      2|engineering|
+-------+-----------+



In [0]:
df_join = df_emp.join(df_dept, on='dept_id', how='inner')
df_join.show()

+-------+------+----+-------+------+-----------+
|dept_id|emp_id|name|surname|salary| department|
+-------+------+----+-------+------+-----------+
|      1|     1|John|    Doe| 60000|  marketing|
|      2|     2|Jane|  Smith| 80000|engineering|
|      1|     3| Sam|  Brown| 75000|  marketing|
|      2|     4|Lisa|    Ray| 90000|engineering|
|      1|     5| Tom|  Hanks| 50000|  marketing|
+-------+------+----+-------+------+-----------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
import pyspark.sql.functions as F

windowspec = Window.partitionBy(df_join.dept_id).orderBy(df_join.salary.desc())
df_ranked = df_join.withColumn("rank", row_number().over(windowspec))
df_ranked.show()

+-------+------+----+-------+------+-----------+----+
|dept_id|emp_id|name|surname|salary| department|rank|
+-------+------+----+-------+------+-----------+----+
|      1|     3| Sam|  Brown| 75000|  marketing|   1|
|      1|     1|John|    Doe| 60000|  marketing|   2|
|      1|     5| Tom|  Hanks| 50000|  marketing|   3|
|      2|     4|Lisa|    Ray| 90000|engineering|   1|
|      2|     2|Jane|  Smith| 80000|engineering|   2|
+-------+------+----+-------+------+-----------+----+



In [0]:
df_filter = df_ranked.filter(df_ranked.rank == 1 )
df_filter.show()

+-------+------+----+-------+------+-----------+----+
|dept_id|emp_id|name|surname|salary| department|rank|
+-------+------+----+-------+------+-----------+----+
|      1|     3| Sam|  Brown| 75000|  marketing|   1|
|      2|     4|Lisa|    Ray| 90000|engineering|   1|
+-------+------+----+-------+------+-----------+----+



In [0]:
result = df_filter.select(
    F.max(F.when(F.col("department") == "marketing", F.col("salary"))).alias("marketing_salary"),
    F.max(F.when(F.col("department") == "engineering", F.col("salary"))).alias("engineering_salary")
).withColumn(
    "salary_diff",
    F.abs(F.col("marketing_salary") - F.col("engineering_salary"))
)

result.show()

+----------------+------------------+-----------+
|marketing_salary|engineering_salary|salary_diff|
+----------------+------------------+-----------+
|           75000|             90000|      15000|
+----------------+------------------+-----------+

